# Semantic joins: evidence for official-statistics workflows

**Contextual selection improved historical affiliation matching. Product matching recovered more links but also created more false links.** These are useful, different findings; neither establishes an operational national statistical office (NSO) workflow.

This notebook keeps the measured results, actual examples and next decision together. A **semantic join** links records when they satisfy a relation stated in language—for example, an affiliation identifies an organization, or a description meets a classification definition. These experiments use a custom candidate selector; the [companion primer](record-linkage-with-semantic-operators.ipynb) demonstrates native LOTUS operators.

All tables replay saved evidence without model calls. [Methods and reproduction](nso-semantic-workflows/README.md).

In [1]:
from IPython.utils.capture import capture_output
with capture_output():
    from pathlib import Path
    import sys, json, hashlib, contextlib, io
    import pandas as pd
    from IPython.display import display, Markdown, HTML

    ROOT = next(p.resolve() for p in [Path('nso-semantic-workflows'), Path('LLM_basics/nso-semantic-workflows')]
                if (p / 'analyze.py').exists())
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from verify import verify_ledger
    report = json.loads((ROOT / 'results/analysis/summary.json').read_text())
    assert report['statistics']['status'] == 'complete'
    audit = verify_ledger()
    assert audit['unresolved_attempts'] == 0
    assert audit['attempted_calls'] == report['ledger_snapshot']['attempted_calls']
    for name, expected in report['inputs_sha256'].items():
        assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name

    def table(rows):
        frame = pd.DataFrame(rows)
        styles = ('<style>.jp-RenderedHTMLCommon .nso-table table {font-size:14px; border-collapse:collapse; max-width:100%;} '
                  '.jp-RenderedHTMLCommon .nso-table td,.jp-RenderedHTMLCommon .nso-table th {padding:8px 10px; text-align:left; vertical-align:top; white-space:normal;} '
                  '.jp-RenderedHTMLCommon .nso-table th {border-bottom:1px solid #c7d1db;} </style>')
        display(HTML(styles + '<div class="nso-table">' +
                     frame.to_html(index=False, escape=True, border=0) + '</div>'))

    display(HTML('<style>.jp-RenderedHTMLCommon table {font-size:14px; width:100%;} '
                 '.jp-RenderedHTMLCommon td,.jp-RenderedHTMLCommon th {padding:8px 10px; text-align:left; vertical-align:top; white-space:normal;} '
                 '.jp-RenderedHTMLCommon blockquote {overflow-wrap:anywhere;} </style>'))
    print(f"Verified {audit['attempted_calls']} recorded model calls and complete result inputs; no new calls.")
    import subprocess
    followup_audit = json.loads(subprocess.check_output([sys.executable, str(ROOT / 'adversarial-2026/verify_replay.py')], text=True))
    assert followup_audit['status'] == 'verified'

    FOLLOWUP = ROOT / 'adversarial-2026'
    FOLLOWUP_HASHES = {'products/results/model-development-v1/conventional-primary.json': 'd8a25d37f0a9802d41a995bdcb188b20c06f6d994aa75196c3fe0c08b057772e', 'products/results/model-development-v1/semantic-primary.json': '0bbce3b833cd8e7bf397eb3b68afe02b32ebae66fdfa75693a35af03048b9269', 'images/results/model-development-v1/comparison.json': '8fe51e49767f960d7fdc071b1dafbeea6a0e0baa22d9790dcd48f8379ad76895', 'images/results/free-test-summary.json': '1c1996b638496a45ffce112a3f6d87673e61fd644a4d3b75a4f0e10c3575c6b2', 'organizations/results/development-score.json': 'ed1b2adafa32684484171131752a0acb17a51967a56374d011b30801868919d8'}
    def saved_followup(name):
        raw = (FOLLOWUP / name).read_bytes()
        assert hashlib.sha256(raw).hexdigest() == FOLLOWUP_HASHES[name], name
        return json.loads(raw)

    product_conventional = saved_followup('products/results/model-development-v1/conventional-primary.json')
    product_semantic = saved_followup('products/results/model-development-v1/semantic-primary.json')
    image_development = saved_followup('images/results/model-development-v1/comparison.json')
    image_free_test = saved_followup('images/results/free-test-summary.json')
    organization_development = saved_followup('organizations/results/development-score.json')

    def decision_row(label, s):
        return {'Method': label, 'Correct / total': f"{s['correct_complete_decisions']}/{s['queries']}",
                'Automatic': s['automatic_decisions'], 'False links': s['false_links'],
                'Missed links': s['missed_links'], 'Review': s['review_queries'], 'Failed': s['failed_queries']}

    table([decision_row(label, group[key]['summary']) for label, group, key in [
        ('Lexical, fixed 0.65', product_conventional, 'lexical_all_pairs'),
        ('Splink, fixed 0.9', product_conventional, 'splink_all_pairs'),
        ('Recordwise extraction → Splink', product_semantic, 'extraction_splink_all_pairs'),
        ('Model candidate selection', product_semantic, 'semantic_selection')]])

    from pathlib import Path as _ph_Path
    import json as _ph_json, hashlib as _ph_hashlib
    import pandas as _ph_pd
    from IPython.display import display as _ph_display, Markdown as _ph_Markdown

    _ph_roots = [_ph_Path('nso-semantic-workflows/adversarial-2026'),
                 _ph_Path('LLM_basics/nso-semantic-workflows/adversarial-2026')]
    _ph_root = next(p.resolve() for p in _ph_roots if (p/'products/score_evaluation.py').is_file())
    _ph_output = _ph_root/'products/results/model-evaluation-v1'
    _ph_ready = (_ph_output/'completion.json').is_file()
    assert _ph_ready, 'Missing sealed product evaluation artifacts'
    _PH_COMPLETION_SHA256 = '9c79f59100858df47454bb36f0f47c0d4fef9ab761f03f5ee93833231c194b41'
    _ph_table = globals().get('table', lambda rows: _ph_display(_ph_pd.DataFrame(rows)))

    def _ph_read(path):
        value = _ph_json.loads(path.read_text())
        _ph_json.dumps(value, allow_nan=False)
        return value

    def _ph_hash(path):
        h = _ph_hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024*1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def _ph_bound(name):
        path = (_ph_output/name).resolve()
        assert path.is_relative_to(_ph_output), name
        assert name in _ph_completion['files'] and _ph_hash(path) == _ph_completion['files'][name], name
        return _ph_read(path)

    _ph_primary_keys = [
        ('Lexical: all pairs', 'lexical_all_pairs', 'lexical/threshold=0.65/all_pairs'),
        ('FS: all pairs', 'splink_all_pairs', 'splink/assumed_recall=0.8;threshold=0.9/all_pairs'),
        ('Lexical: same candidates', 'lexical_same_candidates', 'lexical/threshold=0.65/same_candidates'),
        ('FS: same candidates', 'splink_same_candidates', 'splink/assumed_recall=0.8;threshold=0.9/same_candidates'),
        ('Semantic selection', 'semantic_selection', 'semantic_selection'),
    ]
    if not _ph_ready:
        _ph_display(_ph_Markdown('**Pending sealed evaluation.** No held-out result is loaded or implied.'))
    else:
        assert _ph_hash(_ph_output/'completion.json') == _PH_COMPLETION_SHA256
        _ph_completion = _ph_read(_ph_output/'completion.json')
        assert _ph_completion['status'] == 'sealed_held_out_tradeoff_scored_once'
        assert _ph_hash(_ph_root/'products/score_evaluation.py') == _ph_completion['evaluator_sha256']
        for _ph_name, _ph_digest in _ph_completion['files'].items():
            _ph_path = (_ph_output/_ph_name).resolve()
            assert _ph_path.is_relative_to(_ph_output) and _ph_hash(_ph_path) == _ph_digest, _ph_name
        _ph_seal_path = _ph_root/'results/product-holdout-v1/prediction-seal.json'
        assert _ph_hash(_ph_seal_path) == _ph_completion['seal_file_sha256']
        _ph_seal = _ph_read(_ph_seal_path)
        assert _ph_seal['status'] == 'predictions_sealed'
        assert _ph_seal['manifest_sha256'] == _ph_completion['manifest_sha256']
        assert _ph_seal['evaluator_sha256'] == _ph_completion['evaluator_sha256']
        _ph_manifest = _ph_read(_ph_root/'contracts/products-tradeoff-test-v1.json')
        _ph_manifest_bytes = _ph_json.dumps(_ph_manifest, ensure_ascii=False, sort_keys=True,
                                          separators=(',', ':'), allow_nan=False).encode()
        assert _ph_hashlib.sha256(_ph_manifest_bytes).hexdigest() == _ph_completion['manifest_sha256']
        _ph_predictions = (_ph_root/_ph_seal['predictions_path']).resolve()
        assert _ph_predictions.is_relative_to(_ph_root)
        assert _ph_hash(_ph_predictions) == _ph_seal['predictions_sha256']
        _ph_summary = _ph_bound('primary-summary.json')
        _ph_all = _ph_bound('all-fixed-rule-summaries.json')
        _ph_paired = _ph_bound('paired-family-uncertainty.json')
        _ph_candidates = _ph_bound('candidate-audit.json')
        _ph_subgroups = _ph_bound('subgroup-summaries.json')
        _ph_loss = _ph_bound('descriptive-loss-grid.json')
        assert _ph_summary['results'] == _ph_completion['primary_summary']
        assert len(_ph_all) == 65 and set(_ph_paired) == set(_ph_all)-{'semantic_selection'}
        assert _ph_summary['queries'] == _ph_seal['query_count'] == _ph_seal['terminal_result_count']
        assert all(v['queries'] == _ph_summary['queries'] for v in _ph_all.values())
        assert len({v['query_denominator_sha256'] for v in _ph_all.values()}) == 1
        _ph_execution = _ph_summary['execution_audit']
        _ph_display(_ph_Markdown(
            f"Verified **{_ph_summary['queries']:,} queries**, **{_ph_summary['catalog_records']:,} catalog records**, "
            f"and **{_ph_summary['source_families']:,} query families**. "
            "The original development gate remains failed; extraction remains development-only."))
        _ph_table([
            {'Execution / cost item':'Provider terminal state', 'Recorded value':_ph_execution['terminal_status']},
            {'Execution / cost item':'Attempts / terminal results', 'Recorded value':f"{_ph_execution['attempted_requests']} / {_ph_execution['terminal_result_count']}"},
            {'Execution / cost item':'Parsed result states', 'Recorded value':str(_ph_execution['parsed_result_status_counts'])},
            {'Execution / cost item':'Decision states', 'Recorded value':str(_ph_execution['decision_status_counts'])},
            {'Execution / cost item':'Returned model identifiers', 'Recorded value':str(_ph_execution['provider_model_id_counts'])},
            {'Execution / cost item':'Reserved USD', 'Recorded value':f"{float(_ph_execution['reserved_usd']):.4f}"},
            {'Execution / cost item':'Known token-price estimate, USD', 'Recorded value':f"{float(_ph_execution['nominal_token_price_usd']):.4f}"},
            {'Execution / cost item':'Unpriced results', 'Recorded value':_ph_execution['unpriced_results']},
        ])
        _ph_display(_ph_Markdown('Token-price estimates are not invoices. Any unpriced results make the known usage sum incomplete.'))

## 1. The strongest positive: affiliation text → organization

[S2AFF's published test annotations](https://github.com/allenai/S2AFF/tree/ac8d6b58f42b253821c26fff8b013913317c5c72) link 644 historical affiliation strings to complete sets of Research Organization Registry (ROR) identifiers. Retrieval searches 102,742 registry entries; GPT-6 Luna selects from 25 candidates per string. The lexical comparator uses names, aliases and location evidence, with thresholds selected on separate validation records.

**Valid agreement with the annotated set improves from 456 to 593/644, while false target assignments fall from 174 to 30.** This is evidence that contextual selection helps this implementation. A trained S2AFF reranker was not evaluated.

In [2]:
l = report['linkage']['test']
table([
    {'Measure': 'Valid annotated sets', 'Lexical': l['baseline_correct'], 'Semantic selection': l['model_correct']},
    {'Measure': 'False target assignments', 'Lexical': l['baseline_false_target_assignments'], 'Semantic selection': l['false_target_assignments']},
    {'Measure': 'Corrections / regressions against lexical', 'Lexical': '—', 'Semantic selection': f"{l['corrected']} / {l['regressed']}"},
    {'Measure': 'Correct among unflagged automatic outputs', 'Lexical': '—', 'Semantic selection': f"{l['automatic_correct']} / {l['automatic_count']}"},
])

Measure,Lexical,Semantic selection
Valid annotated sets,456,593
False target assignments,174,30
Corrections / regressions against lexical,—,152 / 15
Correct among unflagged automatic outputs,—,546 / 576


<a id="dataset-showcase"></a>
The 593 agreements include **47 correct selections flagged for review**. They are not 593 automatic successes: 30 of 576 unflagged outputs are wrong. Nine positive rows lack a complete candidate set. The historical annotations and public research organizations also differ from an NSO business register.

These are the first saved correction and incorrect automatic selection by case ID. They illustrate both useful interpretation and an entity-boundary error; the complete test determines the result.

In [3]:
import html
def target_names(items):
    return '; '.join(x.get('name') or x['id'] for x in items) or 'No target'
for category,label in [('corrected','A corrected selection'),('wrong_automatic','An incorrect automatic selection')]:
    x=report['linkage']['test']['examples'][category]
    display(HTML(f"<p><b>{label} — {x['id']}</b></p><blockquote>{html.escape(x['text'])}</blockquote>"))
    table([{'Reference':target_names(x['reference_targets']), 'Lexical':target_names(x['baseline_targets']), 'Model':target_names(x['model_targets'])}])

Reference,Lexical,Model
Xidian University,No target,Xidian University


Reference,Lexical,Model
Peking University,Institute for Nuclear Science and Technology,Peking University; Institute of Software


<a id="product-holdout"></a>
## 2. The counterexample: recovery can damage rejection

The Amazon–Google product holdout contains **1,219 queries and 2,874 catalog records**, with 231 queries that have no publisher-supplied counterpart. The model selects from retrieved candidates. An exact decision must return every reference target and no extra target; review and invalid responses remain unresolved in the denominator.

The frozen lexical rule uses threshold 0.65. Splink uses unlabeled Fellegi–Sunter fitting, assumed anchor recall 0.8 and threshold 0.9. These are fixed implementations, not a ranking of every possible conventional matcher. All-pairs and same-candidate conventional results agree at these settings.

In [4]:
policies = [('Lexical', 'lexical_all_pairs', 'lexical/threshold=0.65/all_pairs'),
            ('Splink', 'splink_all_pairs', 'splink/assumed_recall=0.8;threshold=0.9/all_pairs'),
            ('Semantic', 'semantic_selection', 'semantic_selection')]
rows = []
for label, key in [('Exact decisions / 1,219', 'correct_complete_decisions'),
                   ('False links (pairs)', 'false_links'), ('Missed links (pairs)', 'missed_links'),
                   ('Automatic decisions / 1,219', 'automatic_decisions'),
                   ('Review', 'review_queries'), ('Invalid responses', 'failed_queries')]:
    rows.append({'Measure': label, **{name: _ph_summary['results'][short][key] for name, short, _ in policies}})
rows.insert(3, {'Measure': 'Correct no-counterpart decisions / 231', **{
    name: _ph_subgroups[full]['reference_cardinality']['NIL']['correct_complete_decisions']
    for name, _, full in policies}})
table(rows)

Measure,Lexical,Splink,Semantic
"Exact decisions / 1,219",382,275,702
False links (pairs),264,45,508
Missed links (pairs),788,1082,151
Correct no-counterpart decisions / 231,152,216,27
"Automatic decisions / 1,219",1219,1219,1125
Review,0,0,68
Invalid responses,0,0,26


The selector makes **482 corrections and 162 regressions** against lexical: +26.3 percentage points in exact decisions, with a paired source-family 95% interval of 22.0–30.6 points. Its 702 exact decisions exceed all 64 predeclared conventional policies in the [fixed grid](nso-semantic-workflows/adversarial-2026/products/results/model-evaluation-v1/all-fixed-rule-summaries.json). The recovery gain is real against those comparisons.

The rejection failure is equally material: the selector links 173 of 231 no-counterpart queries and correctly rejects only 27, versus 152 for lexical. Of its 1,125 automatic decisions, 423 are wrong. “No counterpart” means absent from the publisher mapping; annotation omissions remain possible. Those disagreements remain errors under the unchanged reference.

<a id="product-access-chronology"></a>
The original development gate failed. A documented one-time amendment permitted this unchanged holdout. After request submission but before prediction sealing, editorial inspection exposed reference-derived metadata for two held-out records. No prompts, candidates or rules changed; all predictions were sealed before full scoring. [Protocol, access chronology and complete outcomes](nso-semantic-workflows/adversarial-2026/products/README.md#scored-comparisons).

## 3. What the other tests add

The coding descriptions are **authored synthetic cases without independent human labels**. Their gains illustrate interpretation of classification boundaries; they do not estimate accuracy on real returns. Industry uses the North American Industry Classification System (NAICS); occupation uses the National Occupational Classification (NOC). Both search the full official indexes before candidate selection.

In [5]:
rows = []
for scheme, label in [('naics', 'Industry coding'), ('noc', 'Occupation coding')]:
    s = report['coding']['test']['schemes'][scheme]
    rows.append({'Task': label, 'Conventional': f"{s['baseline_correct']}/{s['n']}",
                 'Model': f"{s['model_correct']}/{s['n']}", 'Meaning': 'Synthetic boundary cases'})
rows += [
    {'Task': 'School continuity', 'Conventional': '300/300', 'Model': 'Not run', 'Meaning': 'Ordinary probability sample; model spending stopped'},
    {'Task': 'Catalog source selection', 'Conventional': '9/24', 'Model': '15/24', 'Meaning': 'Reached fixed shortlist ceiling; 9 targets not retrieved'},
    {'Task': 'Numerical questions', 'Conventional': '29/32', 'Model': '29/32 one-shot and iterative', 'Meaning': 'Iteration added no net correct answers'},
    {'Task': 'Package-image association', 'Conventional': '63/80 (CLIP)', 'Model': '63/80 (pixel extraction + lexical)', 'Meaning': 'Development only; model holdout gate failed'},
    {'Task': 'Product extraction → Splink', 'Conventional': '56/144 (lexical)', 'Model': '27/144', 'Meaning': 'Development only; gate failed'},
]
table(rows)

Task,Conventional,Model,Meaning
Industry coding,18/32,28/32,Synthetic boundary cases
Occupation coding,23/32,27/32,Synthetic boundary cases
School continuity,300/300,Not run,Ordinary probability sample; model spending stopped
Catalog source selection,9/24,15/24,Reached fixed shortlist ceiling; 9 targets not retrieved
Numerical questions,29/32,29/32 one-shot and iterative,Iteration added no net correct answers
Package-image association,63/80 (CLIP),63/80 (pixel extraction + lexical),Development only; model holdout gate failed
Product extraction → Splink,56/144 (lexical),27/144,Development only; gate failed


These first corrected coding cases show the source description and resulting join. The complete 32-case test for each scheme includes eight explicitly incomplete cases requiring review; the model requests review on all eight in each scheme. Such explicit missingness is easier than natural ambiguity.

In [6]:
for scheme,label in [('naics','Establishment activity'),('noc','Individual occupation')]:
    x=report['coding']['test']['schemes'][scheme]['examples']['corrected']
    source=next(e for e in x['source_label']['source_evidence'] if e['code']==x['reference']['code'])
    display(HTML(f"<p><b>{label} — {x['id']}</b></p><blockquote>{html.escape(x['text'])}</blockquote>"))
    table([{'Reference':x['reference']['code'] + ' — ' + source['title'],
            'Lexical selection':x['baseline_prediction']['code'],
            'Model selection':x['raw_prediction']['code']}])

Reference,Lexical selection,Model selection
541940 — Veterinary services,622210,541940


Reference,Lexical selection,Model selection
31103 — Veterinarians,32104,31103


The numerical controller uses **1.85×** the one-shot tokens and returns three wrong answers without review, versus one for one-shot planning. The pixel pipeline ties CLIP while reducing automatic coverage from 80 to 71 and false links from 17 to eight. These support stopping those model extensions under their original gates.

The separate changed-name-and-street school sample scores 72/100 conventionally; it is not pooled with the ordinary sample. Later organization and 1,000-image holdouts contain **conventional results only**. The 80 rendered document-control records have OCR outputs but no evaluated linkage result. [All stopped and supporting work](nso-semantic-workflows/adversarial-2026/README.md).

## 4. A government-text relation to inspect

The next priority is substantial government and administrative text. Here is one existing, **unscored illustration** from LePaRD: the context in a later court opinion links to a passage quoted from an earlier opinion. The objects are different documents; the relation is use of a passage in an argument.

The excerpts below come from a source-extracted quotation link. They show the data shape, not a measured model success or a complete set of relevant precedents. Source identifiers and the observed quotation belong to the evaluation/provenance view when testing context-only retrieval. [Complete records, source and licence](nso-semantic-workflows/government-documents/README.md).

In [7]:
government = json.loads((ROOT / 'government-documents/samples.json').read_text())
g = government['examples'][0]['raw_row']
# Display bounded verbatim excerpts; the linked source preserves every field.
table([
    {'Side': 'Query document', 'Source': g['dest_name'], 'Excerpt': g['destination_context'].strip()[:460] + '…'},
    {'Side': 'Observed linked passage', 'Source': g['source_name'], 'Excerpt': g['quote'][:460] + '…'},
])

Side,Source,Excerpt
Query document,"United States v. Associated Credit Bureaus, Inc.","It does not prohibit the members from using other organizations as sources of credit information.\nIf the 1933 decree were equally susceptible to the interpretations the government is now urging, the Court would not accept those interpretations. ACB’s “coupon rules” have been in existence since the establishment of ACB’s parent organization in 1906 and the wording of those rules has not been substantially changed since that time. The consent order of 1953 r…"
Observed linked passage,United States v. Atlantic Refining Co.,"We merely hold that where the language of a consent decree in its normal meaning supports an interpretation; where that interpretation has been adhered to over many years by all the parties, including those government officials who drew up and administered the decree from the start (citation omitted); and where the trial court concludes that this interpretation is in fact the one the parties intended, we will not reject it simply because another reading mi…"


## 5. Next decision and to-dos

**Recommendation: test one document relation with a small source audit before another model experiment.** The candidate is a [registered trial/protocol](https://clinicaltrials.gov/data-api/api) → [publication reporting results from that trial](https://pmc.ncbi.nlm.nih.gov/tools/openftlist/). That could support a research-output coverage file. Its relevance to an actual NSO statistical product still needs establishing; it is a proposed mechanism study, not the next proven NSO use case. Charity-return recipient → charity-register linkage is a more direct administrative alternative, not a second required study.

- [ ] **Define the statistical use.** Name the user, resulting linked dataset, decision it supports, and acceptable false-link/review burden. Select one relation before expanding the data.
- [ ] **Inspect a capped source packet.** Proposed scope: 12 trial families and at most 36 pair inspections, separating ordinary sampled cases from deliberate challenges. Preserve actual text, exclusions, source versions and unresolved labels. Registry references are leads to verify; an unlisted article is not automatically a negative.
- [ ] **Run the inexpensive comparison first.** Retain real identifiers. Try identifier extraction, field matching and retrieval on the same text. Record candidate misses and naturally unresolved cases. Artificially hiding identifiers is a separate capability test.
- [ ] **Make one go/no-go decision.** Proceed only if source access, trustworthy labels and a useful residual relation survive that check. Otherwise close the packet with the reason; do not automatically switch datasets.
- [ ] **Specify the later test before inference.** Freeze grouped development/holdout splits, a capable reranking comparison, error/review limits, all-attempt metrics and budget. Compare correct recovered relations at the same error allowance. Staff-time or statistical-quality claims require their own measurements.

No government-document model experiment has been run here. Further source acquisition and model work are next actions, not prerequisites for reading the completed evidence.

[Initial protocols and paired outcomes](nso-semantic-workflows/README.md#follow-the-evidence) · [Full product evaluation](nso-semantic-workflows/adversarial-2026/products/README.md) · [Short methodological primer](record-linkage-with-semantic-operators.ipynb)